In [6]:
import os, sys
ROOT_DIR = os.path.abspath('..') if os.path.basename(os.getcwd()) in ('visualization', 'notebooks', 'scripts') else os.path.abspath('.')
for _p in (ROOT_DIR, os.path.join(ROOT_DIR, 'src'), os.path.join(ROOT_DIR, 'visualization')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
os.chdir(ROOT_DIR)
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from observables_matrix import *
from bound_labels import compute_bound_stars
from plot_swift_output import load_snapshot

In [2]:
import os
folder = "outputs_SWIFT_jed/withoutDF/"
outputs_folders = [f for f in os.listdir(folder) if f.startswith("IC_")]

# figure 
import cartopy.crs as ccrs

# Collect all snapshot filenames from the first folder (assuming they are consistent across folders)
first_folder_path = folder + outputs_folders[0]
snapshot_files = [f for f in os.listdir(first_folder_path) if f.startswith("snapshot_")]

for snapshot_file in snapshot_files:
    fig = plt.figure(figsize=(5, 5))
    proj_north = ccrs.Orthographic(central_longitude=270.0, central_latitude=30.0)
    ax1 = fig.add_subplot(111, projection=proj_north)
    ax1.set_global() 
    gl1 = ax1.gridlines(draw_labels=False, color='black', alpha=0.4, linestyle='-')

    for folder_name in outputs_folders:
        path = folder + folder_name
        fname = os.path.join(path, snapshot_file)
        
        if os.path.exists(fname): # Check if the snapshot exists in this folder
            pos_raw, vel_raw, _, _ = load_snapshot(fname, shift=500.0)

            x, y, z = pos_raw[:, 0] * u.kpc, pos_raw[:, 1] * u.kpc, pos_raw[:, 2] * u.kpc
            
            coords = SkyCoord(x=x, y=y, z=z, frame='galactocentric', representation_type='cartesian')
            galactic_coords = coords.galactic
            l = galactic_coords.l
            b = galactic_coords.b

            l = np.array(l)
            b = np.array(b)
            l = l - 180
            
            sc1 = ax1.scatter(l, b, s=1, transform=ccrs.Geodetic())

    # Ensure the output directory exists
    os.makedirs("all_streams_figure", exist_ok=True)
    plt.savefig(f"all_streams_figure/{snapshot_file.split('.')[0]}.png")
    plt.close(fig) # Close the figure to free up memory

In [14]:
# making a simple x-y plot of all the strams in time for snapshot_file in snapshot_files:
# Collect all snapshot filenames from the first folder (assuming they are consistent across folders)
import os
folder = "outputs_SWIFT_jed/withoutDF/"
folder = "/Volumes/Adrian 1 SS/MA2/TP4b/outputs_SWIFT_jed/withoutDF/"
outputs_folders = [f for f in os.listdir(folder) if f.startswith("IC_")]

first_folder_path = folder + outputs_folders[0]
snapshot_files = [f for f in os.listdir(first_folder_path) if f.startswith("snapshot_")]
for snapshot_file in snapshot_files: 
    fig = plt.figure(figsize=(5, 5))
    
    ax1 = fig.add_subplot(111)
    for folder_name in outputs_folders:
        path = folder + folder_name
        fname = os.path.join(path, snapshot_file)
        
        if os.path.exists(fname): # Check if the snapshot exists in this folder
            pos_raw, vel_raw, _, _ = load_snapshot(fname, shift=500.0)

            x, y, z = pos_raw[:, 0] * u.kpc, pos_raw[:, 1] * u.kpc, pos_raw[:, 2] * u.kpc
            ax1.scatter(y, z, s=1)
            ax1.set_xlabel("Y [kpc]")
            ax1.set_ylabel("Z [kpc]")   
            ax1.set_aspect('equal')  
            ax1.text(np.mean(pos_raw[:, 1]), np.mean(pos_raw[:, 2]), folder_name, fontsize=8, ha='center', va='center')   
            ax1.set_xlim(-30, 30)
            ax1.set_ylim(-30, 30)
        else :
            print(f"Snapshot {snapshot_file} not found in folder {folder_name}. Skipping.")

    # Ensure the output directory exists
    os.makedirs("all_streams_figure/y-z", exist_ok=True)
    plt.savefig(f"all_streams_figure/y-z/{snapshot_file.split('.')[0]}.png")
    plt.close(fig) # Close the figure to free up memory

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

folder = "outputs_SWIFT_jed/withoutDF/"
outputs_folders = [f for f in os.listdir(folder) if f.startswith("IC_")]
first_folder_path = folder + outputs_folders[0]
snapshot_files = [f for f in os.listdir(first_folder_path) if f.startswith("snapshot_")]

cmaps = ['Blues', 'Oranges', 'Greens', 'Reds', 'Purples', 'Greys', 'pink', 'copper']

os.makedirs("all_streams_figure/y-z", exist_ok=True)

for snapshot_file in snapshot_files:
    fig = plt.figure(figsize=(6, 5))
    ax1 = fig.add_subplot(111)
    
    all_x, all_y, all_z, all_density, all_colors = [], [], [], [], []
    
    # 1. Collecter les données de tous les streams d'abord
    for i, folder_name in enumerate(outputs_folders):
        fname = os.path.join(folder + folder_name, snapshot_file)
        if os.path.exists(fname):
            pos_raw, _, _, _ = load_snapshot(fname, shift=500.0)
            
            x = pos_raw[:, 0]
            y = pos_raw[:, 1]
            z = pos_raw[:, 2]
            
            # Calcul de densité pour CE stream spécifique
            counts, yedges, zedges = np.histogram2d(y, z, bins=200, range=[[-30, 30], [-30, 30]])
            y_idx = np.clip(np.digitize(y, yedges) - 1, 0, counts.shape[0] - 1)
            z_idx = np.clip(np.digitize(z, zedges) - 1, 0, counts.shape[1] - 1)
            density = counts[y_idx, z_idx]
            
            # Stocker les valeurs
            all_x.append(x)
            all_y.append(y)
            all_z.append(z)
            all_density.append(density)
            
            # Obtenir les couleurs RGBA depuis la colormap pour chaque particule
            cmap = plt.get_cmap(cmaps[i % len(cmaps)])
            # Normaliser la densité entre 0 et 1 pour la colormap
            if density.max() > density.min():
                norm_density = (density - density.min()) / (density.max() - density.min())
            else:
                norm_density = np.zeros_like(density)
            colors = cmap(norm_density)
            # Rendre les zones peu denses transparentes pour mieux voir à travers
            colors[:, 3] = np.clip(norm_density + 0.1, 0, 1) # Canal alpha
            all_colors.append(colors)
            
    if not all_x:
        continue
        
    # 2. Concaténer toutes les données
    x_flat = np.concatenate(all_x)
    y_flat = np.concatenate(all_y)
    z_flat = np.concatenate(all_z)
    colors_flat = np.concatenate(all_colors)
    
    # 3. Trier selon la coordonnée X (profondeur)
    # argsort() trie du plus petit au plus grand.
    # Les particules avec un X petit (au fond) seront tracées en premier.
    # Les particules avec un X grand (devant) seront tracées en dernier (par-dessus).
    depth_order = x_flat.argsort()
    
    y_sorted = y_flat[depth_order]
    z_sorted = z_flat[depth_order]
    colors_sorted = colors_flat[depth_order]
            
    # 4. Tracer en une seule fois avec les couleurs pré-calculées
    ax1.scatter(y_sorted, z_sorted, c=colors_sorted, s=1, edgecolor='none')
            
    ax1.set_xlabel("Y [kpc]")
    ax1.set_ylabel("Z [kpc]")
    ax1.set_aspect('equal')
    ax1.set_xlim(-30, 30)
    ax1.set_ylim(-30, 30)
    
    plt.savefig(f"all_streams_figure/y-z/{snapshot_file.split('.')[0]}.png", bbox_inches='tight', dpi=150)
    plt.close(fig)

In [4]:
from Stellar_objects.visu import *
make_animation("all_streams_figure/x-y/", "all_streams_figure/x-y/animation.gif", 5)

loading from 101 images...


Creation of all_streams_figure/x-y/animation.gif at 5 FPS...
Done !
